# Distribuição dos tamanhos dos clusters — modelos do LaTeX

Figura para o paper com um painel por dataset e um violino por modelo. A lista e a ordem dos modelos são extraídas **do arquivo LaTeX**, não da seleção automática maior. Assim, os modelos removidos da tabela também ficam fora da figura.

Cada observação corresponde ao **tamanho de um cluster como percentual de todos os nós do grafo**. Usamos as mesmas runs de `selected_attempts.csv`, lendo seus `labels.npy`. Nenhum modelo é treinado e nenhuma entrada de `results_old/` é consultada.

O adaptive fica por último, em verde; os outros modelos usam uma cor neutra. Círculos brancos mostram a mediana e segmentos escuros mostram o intervalo interquartil. As contagens à direita indicam runs utilizadas/esperadas.

## Configuração

`LATEX_PATH` aponta para a tabela interna atual, com os mesmos modelos da externa. Para atualizar o conjunto e as runs após novos treinos, reexecute primeiro `adaptive_benchmark_comparison.ipynb` e depois este notebook.

`SCALE = "log"` acomoda simultaneamente clusters pequenos e grandes; os rótulos continuam sendo percentuais reais. A KDE é estimada no espaço log10 e limitada ao intervalo observado. Use `"linear"` para comparar na escala linear. PDF e SVG mantêm o desenho vetorial; PNG é salvo a 400 dpi.

In [ ]:
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "cluster_size_violin.py").is_file():
            return candidate
    raise FileNotFoundError("Abra este notebook dentro do repositório.")


ROOT = find_repo_root()
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import cluster_size_violin as violin
importlib.reload(violin)

LATEX_PATH = ROOT / "results/adaptive_benchmark_comparison/top3_subset/internal/comparison_vs_densgnn.tex"
EXPORT_DIR = ROOT / "results/adaptive_benchmark_comparison/top3_subset/cluster_size_violins"
SCALE = "log"
BW_ADJUST = 0.7
FIGSIZE = (7.6, 6.4)

## Ler e validar as distribuições

- Só entram runs bem-sucedidas da seleção salva junto ao LaTeX.
- Cada run contribui com peso total igual; dentro dela, cada cluster tem o mesmo peso. Uma run com mais clusters não domina a KDE.
- O rótulo `-1` representa ruído: não vira um cluster, mas seus nós permanecem no denominador dos percentuais.
- São conferidos formato dos labels, quantidade de nós, seed e a variante adaptive nos metadados.
- Arquivos ausentes ou inválidos ficam na auditoria; o gráfico mostra pendências. Uma distribuição constante aparece como linha, sem inventar uma densidade.

A figura é descritiva: tamanho de cluster não é uma nota de qualidade e depende também da quantidade de clusters produzida pelo modelo. Não há testes estatísticos nem afirmação de independência entre os clusters de uma mesma run.

In [ ]:
data = violin.load_cluster_distributions(LATEX_PATH, ROOT)
print(f"Modelos no LaTeX: {len(data.model_order)}")
print(f"Datasets: {len(data.dataset_order)}")
print(f"Observações de clusters: {len(data.clusters)}")
display(pd.DataFrame({"model": data.model_order, "name": [violin.MODEL_NAMES.get(m, m) for m in data.model_order]}))
display(data.coverage)
issues = data.runs.loc[~data.runs["status"].eq("loaded")]
if not issues.empty:
    display(Markdown("### Runs indisponíveis, sem clusters ou sem sucesso"))
    display(issues[["dataset", "model", "seed", "status", "detail", "noise_percent"]])

## Figura principal

A largura é normalizada separadamente em cada violino e não representa a quantidade de clusters. A mediana/IQR usam os mesmos pesos por run da distribuição. O eixo horizontal é compartilhado entre datasets para permitir comparação visual dos tamanhos relativos.

In [ ]:
figure = violin.plot_cluster_size_violins(
    data,
    scale=SCALE,
    bw_adjust=BW_ADJUST,
    figsize=FIGSIZE,
)
display(figure)
plt.close(figure)

## Quantidade de clusters e ruído

Esta tabela ajuda a interpretar os violinos: partições com muitos clusters tendem a ter tamanhos relativos menores. As estatísticas abaixo usam as runs com labels carregados, incluindo as que contêm somente ruído.

In [ ]:
loaded = data.runs.loc[data.runs["status"].isin(["loaded", "noise_only"])]
if not loaded.empty:
    descriptive = loaded.groupby(["dataset", "model"], sort=False).agg(
        runs=("seed", "count"),
        clusters_median=("n_clusters", "median"),
        clusters_min=("n_clusters", "min"),
        clusters_max=("n_clusters", "max"),
        largest_cluster_percent_median=("largest_cluster_percent", "median"),
        noise_percent_mean=("noise_percent", "mean"),
    ).reset_index()
    display(descriptive)
else:
    print("Nenhuma run com labels válidos disponível neste retrato.")

## Exportar para o paper

Os arquivos `cluster_size_violins.pdf` e `.svg` são vetoriais. O `.png` é uma prévia a 400 dpi. Também são salvos os tamanhos usados na figura, um resumo por run, contagens de cobertura, uma sugestão de legenda em inglês e um manifesto com caminhos e hashes dos arquivos fonte.

In [ ]:
violin.export_cluster_distributions(
    data,
    figure,
    EXPORT_DIR,
    scale=SCALE,
    bw_adjust=BW_ADJUST,
)
if "descriptive" in globals():
    descriptive.to_csv(EXPORT_DIR / "descriptive_summary.csv", index=False)
print(f"Arquivos salvos em: {EXPORT_DIR}")
for path in sorted(EXPORT_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name}")
print("\nSugestão de legenda para o paper:")
print((EXPORT_DIR / "figure_caption.txt").read_text())